### Running R code in a Python Notebook using `rpy2`

To run R code directly within a Python notebook, you can use the `rpy2` library, which allows for seamless interoperability between Python and R. Follow these steps:

In [ ]:
# Load the rpy2.ipython extension
%load_ext rpy2.ipython

Once the extension is loaded, you can use the `%%R` magic command at the beginning of a cell to indicate that the code within that cell should be executed as R code.

In [ ]:
%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


In [ ]:
%%R
install.packages(c("httr", "rvest", "xml2"))

library(httr)
library(rvest)
library(xml2)

print("R packages installed and loaded successfully!")

[1] "R packages installed and loaded successfully!"


Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
trying URL 'https://cran.rstudio.com/src/contrib/httr_1.4.8.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/rvest_1.0.5.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/xml2_1.5.2.tar.gz'

The downloaded source packages are in
	‘/tmp/RtmpJbp81W/downloaded_packages’


This approach is useful when you need to leverage specific R packages or functions within your Python-centric workflow. Alternatively, if your entire notebook is intended to be in R, you should change the runtime type as previously instructed.

In [ ]:
%%R
url <- "https://en.wikipedia.org/wiki/COVID-19_testing"

response <- GET(url)

# Check status code
status_code(response)

[1] 200


#Task 2: Extract COVID-19 Testing Data Table from the Wiki HTML Page

#Get HTML Content

In [ ]:
%%R
html_content <- content(response, as = "text", encoding = "UTF-8")

cat(substr(html_content, 1, 500))

<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientp

#Get the Root HTML Node from the HTTP Response

In [ ]:
%%R
root_html <- read_html(html_content)

root_html

{html_document}
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-sticky-header-enabled vector-toc-available skin-thumbsize-clientpref-standard" lang="en" dir="ltr">
[1] <head>\n<meta http-equiv="Content-Type" content="text/html; charset=UTF-8 ...
[2] <body class="skin--responsive skin-vector skin-vector-search-vue mediawik ...


#Get All Table Nodes from the Root HTML

In [ ]:
%%R
tables <- html_elements(root_html, "table")

length(tables)

[1] 49


#Inspect Available Tables

In [ ]:
%%R
for(i in seq_along(tables)){
  cat("\nTable:", i, "\n")
  print(html_table(tables[[i]], fill = TRUE)[1:5, ])
}


Table: 1 
# A tibble: 5 × 1
  X1                                                                            
  <chr>                                                                         
1 "Part of a series on the"                                                     
2 "COVID-19 pandemic"                                                           
3 "Scientifically accurate atomic model of the external structure of SARS-CoV-2…
4 "COVID-19 (disease)\nSARS-CoV-2 (virus)Cases\nDeaths"                         
5 "Timeline\n20192020January\nresponses\nFebruary\nresponses\nMarch\nresponses\…

Table: 2 
# A tibble: 5 × 2
  `Samples source`                       `Positive rate`
  <chr>                                  <chr>          
1 Bronchoalveolar lavage fluid specimens 93% (14/15)    
2 Sputum                                 72% (75/104)   
3 Nasal swabs                            63% (5/8)      
4 Fibrobronchoscope brush biopsy         46% (6/13)     
5 Pharyngeal swabs               

Select the COVID-19 Testing Data Table

Wikipedia pages may contain multiple tables. Choose the relevant table.

For example:

In [ ]:
%%R
covid_table <- html_table(tables[[3]], fill = TRUE)

head(covid_table)

# A tibble: 6 × 9
  `Country or region` `Date[a]`   Tested  `Units[b]` `Confirmed(cases)`
  <chr>               <chr>       <chr>   <chr>      <chr>             
1 Afghanistan         17 Dec 2020 154,767 samples    49,621            
2 Albania             18 Feb 2021 428,654 samples    96,838            
3 Algeria             2 Nov 2020  230,553 samples    58,574            
4 Andorra             23 Feb 2022 300,307 samples    37,958            
5 Angola              2 Feb 2021  399,228 samples    20,981            
6 Antigua and Barbuda 6 Mar 2021  15,268  samples    832               
# ℹ 4 more variables: `Confirmed /tested,%` <chr>,
#   `Tested /population,%` <chr>, `Confirmed /population,%` <chr>, Ref. <chr>


#Read the Table Node and Convert It Into a Data Frame

In [ ]:
%%R
covid_df <- as.data.frame(covid_table)

head(covid_df)

    Country or region     Date[a]  Tested Units[b] Confirmed(cases)
1         Afghanistan 17 Dec 2020 154,767  samples           49,621
2             Albania 18 Feb 2021 428,654  samples           96,838
3             Algeria  2 Nov 2020 230,553  samples           58,574
4             Andorra 23 Feb 2022 300,307  samples           37,958
5              Angola  2 Feb 2021 399,228  samples           20,981
6 Antigua and Barbuda  6 Mar 2021  15,268  samples              832
  Confirmed /tested,% Tested /population,% Confirmed /population,%       Ref.
1                32.1                 0.40                    0.13      [248]
2                22.6                 15.0                     3.4      [249]
3                25.4                 0.53                    0.13 [250][251]
4                12.6                  387                    49.0      [252]
5                 5.3                  1.3                   0.067      [253]
6                 5.4                 15.9              

#Print the Data Frame for Review

In [ ]:
%%R
print(covid_df)

str(covid_df)

summary(covid_df)

#Clean Column Names

In [ ]:
%%R
colnames(covid_df) <- make.names(colnames(covid_df))

names(covid_df)

[1] "Country.or.region"       "Date.a."                
[3] "Tested"                  "Units.b."               
[5] "Confirmed.cases."        "Confirmed..tested.."    
[7] "Tested..population.."    "Confirmed..population.."
[9] "Ref."                   


#Basic Exploratory Analysis

Number of Rows and Columns

In [17]:
%%R
dim(covid_df)

[1] 173   9


#Missing Values

In [18]:
%%R
colSums(is.na(covid_df))

      Country.or.region                 Date.a.                  Tested 
                      0                       0                       0 
               Units.b.        Confirmed.cases.     Confirmed..tested.. 
                      0                       0                       0 
   Tested..population.. Confirmed..population..                    Ref. 
                      0                       0                       0 


#First 10 Records

In [19]:
%%R
head(covid_df, 10)

     Country.or.region     Date.a.      Tested Units.b. Confirmed.cases.
1          Afghanistan 17 Dec 2020     154,767  samples           49,621
2              Albania 18 Feb 2021     428,654  samples           96,838
3              Algeria  2 Nov 2020     230,553  samples           58,574
4              Andorra 23 Feb 2022     300,307  samples           37,958
5               Angola  2 Feb 2021     399,228  samples           20,981
6  Antigua and Barbuda  6 Mar 2021      15,268  samples              832
7            Argentina 16 Apr 2022  35,716,069  samples        9,060,495
8              Armenia 29 May 2022   3,099,602  samples          422,963
9            Australia  9 Sep 2022  78,548,492  samples       10,112,229
10             Austria  1 Feb 2023 205,817,752  samples        5,789,991
   Confirmed..tested.. Tested..population.. Confirmed..population..       Ref.
1                 32.1                 0.40                    0.13      [248]
2                 22.6                 

#Project Outcome

Retrieved a live COVID-19 Wikipedia page using an HTTP GET request.
Parsed HTML content into a structured document.


Extracted table data from the webpage.
Converted the table into an R data frame.
Performed preliminary data inspection and exploratory analysis.


This workflow demonstrates web scraping, HTTP requests, HTML parsing, and data extraction skills commonly used in data analytics and data science projects.

#TASK 3: Pre-process and export the extracted data frame

#Step 1: Inspect the Data

In [20]:
%%R
# View first few rows
head(covid_df)

# Check structure
str(covid_df)

# Summary statistics
summary(covid_df)

'data.frame':	173 obs. of  9 variables:
 $ Country.or.region      : chr  "Afghanistan" "Albania" "Algeria" "Andorra" ...
 $ Date.a.                : chr  "17 Dec 2020" "18 Feb 2021" "2 Nov 2020" "23 Feb 2022" ...
 $ Tested                 : chr  "154,767" "428,654" "230,553" "300,307" ...
 $ Units.b.               : chr  "samples" "samples" "samples" "samples" ...
 $ Confirmed.cases.       : chr  "49,621" "96,838" "58,574" "37,958" ...
 $ Confirmed..tested..    : chr  "32.1" "22.6" "25.4" "12.6" ...
 $ Tested..population..   : chr  "0.40" "15.0" "0.53" "387" ...
 $ Confirmed..population..: chr  "0.13" "3.4" "0.13" "49.0" ...
 $ Ref.                   : chr  "[248]" "[249]" "[250][251]" "[252]" ...
 Country.or.region      Date.a.           Tested          Units.b.   
 Length   : 173    Length   : 173   Length   : 173   Length   : 173  
 N.unique : 173    N.unique : 142   N.unique : 173   N.unique :   4  
 N.blank  :   0    N.blank  :   0   N.blank  :   0   N.blank  :  16  
 Min.nchar:  

#Step 2: Rename Columns

Clean column names by removing spaces and special characters.

In [21]:
%%R
names(covid_df) <- make.names(names(covid_df))

# View updated column names
names(covid_df)

[1] "Country.or.region"       "Date.a."                
[3] "Tested"                  "Units.b."               
[5] "Confirmed.cases."        "Confirmed..tested.."    
[7] "Tested..population.."    "Confirmed..population.."
[9] "Ref."                   


#Step 3: Remove Duplicate Rows

In [22]:
%%R
covid_df <- covid_df[!duplicated(covid_df), ]

# Number of rows after removing duplicates
nrow(covid_df)

[1] 173


#Step 4: Handle Missing Values

Check for missing values:

In [23]:
%%R
colSums(is.na(covid_df))

      Country.or.region                 Date.a.                  Tested 
                      0                       0                       0 
               Units.b.        Confirmed.cases.     Confirmed..tested.. 
                      0                       0                       0 
   Tested..population.. Confirmed..population..                    Ref. 
                      0                       0                       0 


#Replace missing values with NA:

In [24]:
%%R
covid_df[covid_df == ""] <- NA

#Remove rows with excessive missing values:

In [25]:
%%R
covid_df <- na.omit(covid_df)

#Step 7: Convert Numeric Columns

Identify numeric columns and convert them.

Example:

In [26]:
%%R
numeric_cols <- c("Tests",
                  "Positive",
                  "Tests.per.million")

for(col in numeric_cols){
  if(col %in% names(covid_df)){
    covid_df[[col]] <- as.numeric(covid_df[[col]])
  }
}

#Check structure:

In [27]:
%%R
str(covid_df)

'data.frame':	156 obs. of  9 variables:
 $ Country.or.region      : chr  "Afghanistan" "Albania" "Algeria" "Andorra" ...
 $ Date.a.                : chr  "17 Dec 2020" "18 Feb 2021" "2 Nov 2020" "23 Feb 2022" ...
 $ Tested                 : chr  "154,767" "428,654" "230,553" "300,307" ...
 $ Units.b.               : chr  "samples" "samples" "samples" "samples" ...
 $ Confirmed.cases.       : chr  "49,621" "96,838" "58,574" "37,958" ...
 $ Confirmed..tested..    : chr  "32.1" "22.6" "25.4" "12.6" ...
 $ Tested..population..   : chr  "0.40" "15.0" "0.53" "387" ...
 $ Confirmed..population..: chr  "0.13" "3.4" "0.13" "49.0" ...
 $ Ref.                   : chr  "[248]" "[249]" "[250][251]" "[252]" ...
 - attr(*, "na.action")= 'omit' Named int [1:17] 23 28 29 42 45 49 51 64 67 74 ...
  ..- attr(*, "names")= chr [1:17] "23" "28" "29" "42" ...


#Step 8: Standardize Country Names

Remove extra spaces:

In [30]:
%%R
covid_df$Country.or.region <- trimws(covid_df$Country.or.region)

#Convert to title case:

In [32]:
%%R
covid_df$Country.or.region <- tools::toTitleCase(
  tolower(covid_df$Country.or.region)
)

#Step 9: Create Derived Metrics

Positivity Rate

In [91]:
%%R
# Convert 'Tested', 'Confirmed(cases)', and 'Confirmed..population..' to numeric, handling commas
covid_df$Tested <- as.numeric(gsub(",", "", covid_df$Tested))
covid_df$Confirmed.cases. <- as.numeric(gsub(",", "", covid_df$Confirmed.cases.))
covid_df$Confirmed..population.. <- as.numeric(gsub(",", "", covid_df$Confirmed..population..))

# Calculate Positivity_Rate
covid_df$Positivity_Rate <- (covid_df$Confirmed.cases. / covid_df$Tested) * 100

#Tests per Positive Case

In [36]:
%%R
covid_df$Tests_Per_Case <- covid_df$Tested / covid_df$Confirmed.cases.

#Step 10: Sort Data

Highest testing countries:

In [37]:
%%R
covid_df <- covid_df[
  order(covid_df$Tested, decreasing = TRUE),
]

#View top countries:

In [38]:
%%R
head(covid_df, 10)

       Country.or.region     Date.a.    Tested Units.b. Confirmed.cases.
166        United States 29 Jul 2022 929349291  samples         90749469
73                 India  8 Jul 2022 866177937  samples         43585554
165       United Kingdom 19 May 2022 522526476  samples         22232377
135               Russia  6 Jun 2022 295542733  samples         18358459
56          France[f][g] 15 May 2022 272417258  samples         29183646
79                 Italy 16 Mar 2023 269127054  samples         25651205
10               Austria  1 Feb 2023 205817752  samples          5789991
164 United Arab Emirates  1 Feb 2023 198685717  samples          1049537
34              China[c] 31 Jul 2020 160000000    cases            87655
62                Greece 18 Dec 2022 101576831  samples          5548487
    Confirmed..tested.. Tested..population.. Confirmed..population..       Ref.
166                 9.8                  281                    27.4 [431][432]
73                  5.0              

#Step 11: Export Cleaned Dataset to CSV

In [39]:
%%R
write.csv(
  covid_df,
  "covid_testing_cleaned.csv",
  row.names = FALSE
)

#Step 12: Verify Export

In [40]:
%%R
exported_data <- read.csv(
  "covid_testing_cleaned.csv"
)

head(exported_data)

  Country.or.region     Date.a.    Tested Units.b. Confirmed.cases.
1     United States 29 Jul 2022 929349291  samples         90749469
2             India  8 Jul 2022 866177937  samples         43585554
3    United Kingdom 19 May 2022 522526476  samples         22232377
4            Russia  6 Jun 2022 295542733  samples         18358459
5      France[f][g] 15 May 2022 272417258  samples         29183646
6             Italy 16 Mar 2023 269127054  samples         25651205
  Confirmed..tested.. Tested..population.. Confirmed..population..       Ref.
1                 9.8                  281                    27.4 [431][432]
2                 5.0                   63                    31.7 [328][329]
3                 4.3                  774                    32.9      [430]
4                 6.2                  201                    12.5 [396][397]
5                10.7                  417                    44.7      [310]
6                 9.5                  446              

#Outcome

After preprocessing, the dataset is:

- Cleaned of duplicates and missing values.
- Free of Wikipedia reference tags.
- Converted to appropriate numeric data types.
- Enhanced with new analytical metrics (Positivity Rate and Tests per Case).
- Exported as covid_testing_cleaned.csv for further EDA, visualization, or machine learning tasks.

#TASK 4: Get a subset of the extracted data frame

1. Display the First 10 Rows

In [41]:
%%R
head(covid_df, 10)

       Country.or.region     Date.a.    Tested Units.b. Confirmed.cases.
166        United States 29 Jul 2022 929349291  samples         90749469
73                 India  8 Jul 2022 866177937  samples         43585554
165       United Kingdom 19 May 2022 522526476  samples         22232377
135               Russia  6 Jun 2022 295542733  samples         18358459
56          France[f][g] 15 May 2022 272417258  samples         29183646
79                 Italy 16 Mar 2023 269127054  samples         25651205
10               Austria  1 Feb 2023 205817752  samples          5789991
164 United Arab Emirates  1 Feb 2023 198685717  samples          1049537
34              China[c] 31 Jul 2020 160000000    cases            87655
62                Greece 18 Dec 2022 101576831  samples          5548487
    Confirmed..tested.. Tested..population.. Confirmed..population..       Ref.
166                 9.8                  281                    27.4 [431][432]
73                  5.0              

#2. Select Specific Columns

Suppose you only want the country name and total tests:

In [42]:
%%R
names(covid_df)

 [1] "Country.or.region"       "Date.a."                
 [3] "Tested"                  "Units.b."               
 [5] "Confirmed.cases."        "Confirmed..tested.."    
 [7] "Tested..population.."    "Confirmed..population.."
 [9] "Ref."                    "Positivity_Rate"        
[11] "Tests_Per_Case"         


In [43]:
%%R
colnames(covid_df) <- make.names(colnames(covid_df))
names(covid_df)

 [1] "Country.or.region"       "Date.a."                
 [3] "Tested"                  "Units.b."               
 [5] "Confirmed.cases."        "Confirmed..tested.."    
 [7] "Tested..population.."    "Confirmed..population.."
 [9] "Ref."                    "Positivity_Rate"        
[11] "Tests_Per_Case"         


#3. Get Top 10 Countries by Number of Tests

In [44]:
%%R
top10_tests <- covid_df[order(covid_df$Tested, decreasing = TRUE), ]

top10_tests <- head(top10_tests, 10)

top10_tests

       Country.or.region     Date.a.    Tested Units.b. Confirmed.cases.
166        United States 29 Jul 2022 929349291  samples         90749469
73                 India  8 Jul 2022 866177937  samples         43585554
165       United Kingdom 19 May 2022 522526476  samples         22232377
135               Russia  6 Jun 2022 295542733  samples         18358459
56          France[f][g] 15 May 2022 272417258  samples         29183646
79                 Italy 16 Mar 2023 269127054  samples         25651205
10               Austria  1 Feb 2023 205817752  samples          5789991
164 United Arab Emirates  1 Feb 2023 198685717  samples          1049537
34              China[c] 31 Jul 2020 160000000    cases            87655
62                Greece 18 Dec 2022 101576831  samples          5548487
    Confirmed..tested.. Tested..population.. Confirmed..population..       Ref.
166                 9.8                  281                    27.4 [431][432]
73                  5.0              

#4. Filter Countries with More Than 1 Million Tests

In [45]:
%%R
high_testing <- subset(covid_df, Tested > 1000000)

head(high_testing)

    Country.or.region     Date.a.    Tested Units.b. Confirmed.cases.
166     United States 29 Jul 2022 929349291  samples         90749469
73              India  8 Jul 2022 866177937  samples         43585554
165    United Kingdom 19 May 2022 522526476  samples         22232377
135            Russia  6 Jun 2022 295542733  samples         18358459
56       France[f][g] 15 May 2022 272417258  samples         29183646
79              Italy 16 Mar 2023 269127054  samples         25651205
    Confirmed..tested.. Tested..population.. Confirmed..population..       Ref.
166                 9.8                  281                    27.4 [431][432]
73                  5.0                   63                    31.7 [328][329]
165                 4.3                  774                    32.9      [430]
135                 6.2                  201                    12.5 [396][397]
56                 10.7                  417                    44.7      [310]
79                  9.5       

#5. Select Rows and Columns Together

In [46]:
%%R
country_tests <- covid_df[
  covid_df$Tested > 1000000,
  c("Country.or.region", "Tested")
]

head(country_tests)

    Country.or.region    Tested
166     United States 929349291
73              India 866177937
165    United Kingdom 522526476
135            Russia 295542733
56       France[f][g] 272417258
79              Italy 269127054


#6. Extract Specific Countries

For example, USA, India, and Brazil:

In [47]:
%%R
selected_countries <- subset(
  covid_df,
  Country.or.region %in% c("United States", "India", "Brazil")
)

selected_countries

    Country.or.region     Date.a.    Tested Units.b. Confirmed.cases.
166     United States 29 Jul 2022 929349291  samples         90749469
73              India  8 Jul 2022 866177937  samples         43585554
24             Brazil 19 Feb 2021  23561497  samples         10081676
    Confirmed..tested.. Tested..population.. Confirmed..population..       Ref.
166                 9.8                  281                    27.4 [431][432]
73                  5.0                   63                    31.7 [328][329]
24                 42.8                 11.2                     4.8 [274][275]
    Positivity_Rate Tests_Per_Case
166         9.76484      10.240823
73          5.03194      19.873051
24         42.78878       2.337062


#7. Countries with Highest Positivity Rate

In [48]:
%%R
top_positive <- covid_df[
  order(covid_df$Positivity_Rate, decreasing = TRUE),
]

head(top_positive, 10)

    Country.or.region     Date.a.   Tested Units.b. Confirmed.cases.
146          Slovenia  2 Feb 2023  2826117  samples          1322282
24             Brazil 19 Feb 2021 23561497  samples         10081676
105            Mexico 15 Oct 2021 10503678    cases          3749860
70           Honduras 26 Nov 2021  1133782  samples           377859
1         Afghanistan 17 Dec 2020   154767  samples            49621
46            Ecuador 23 Jul 2021  1627189  samples           480720
155         Taiwan[m]  3 Feb 2023 30275725  samples          8622129
116       New Zealand 29 Jan 2023  7757935  samples          2136662
3             Algeria  2 Nov 2020   230553  samples            58574
7           Argentina 16 Apr 2022 35716069  samples          9060495
    Confirmed..tested.. Tested..population.. Confirmed..population..       Ref.
146                46.8                  135                    63.1      [409]
24                 42.8                 11.2                     4.8 [274][275]
1

#8. Create a Random Sample

In [49]:
%%R
set.seed(123)

sample_data <- covid_df[
  sample(nrow(covid_df), 10),
]

sample_data

#9. Using dplyr (Alternative Method)

In [50]:
%%R
library(dplyr)

covid_subset <- covid_df %>%
  select(Country.or.region, Tested, Positivity_Rate) %>% # Corrected 'Tests' to 'Tested'
  arrange(desc(Tested)) %>% # Corrected 'Tests' to 'Tested'
  slice(1:10)

covid_subset

      Country.or.region    Tested Positivity_Rate
1         United States 929349291      9.76483975
2                 India 866177937      5.03193999
3        United Kingdom 522526476      4.25478479
4                Russia 295542733      6.21177818
5          France[f][g] 272417258     10.71284772
6                 Italy 269127054      9.53126214
7               Austria 205817752      2.81316405
8  United Arab Emirates 198685717      0.52823978
9              China[c] 160000000      0.05478438
10               Greece 101576831      5.46235489



Attaching package: ‘dplyr’

The following objects are masked from ‘package:stats’:

    filter, lag

The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



#Export the Subset

In [51]:
%%R
write.csv(
  covid_subset,
  "covid_subset.csv",
  row.names = FALSE
)

#Expected Outcome

The resulting subset contains:

- Country/Region
- Total Tests
- Positive Cases
- Positivity Rate

for the top 10 countries by testing volume, making it easier to perform focused analysis and visualization in later tasks.

#TASK 5: Calculate worldwide COVID testing positive ratio


The goal of task 5 is to get the total confirmed and tested cases worldwide, and try to figure the overall positive ratio using confirmed cases / tested cases

#Step 1: Inspect the Data

First, ensure the relevant columns are numeric.

In [52]:
%%R
str(covid_df)

'data.frame':	156 obs. of  11 variables:
 $ Country.or.region      : chr  "United States" "India" "United Kingdom" "Russia" ...
 $ Date.a.                : chr  "29 Jul 2022" "8 Jul 2022" "19 May 2022" "6 Jun 2022" ...
 $ Tested                 : num  9.29e+08 8.66e+08 5.23e+08 2.96e+08 2.72e+08 ...
 $ Units.b.               : chr  "samples" "samples" "samples" "samples" ...
 $ Confirmed.cases.       : num  90749469 43585554 22232377 18358459 29183646 ...
 $ Confirmed..tested..    : chr  "9.8" "5.0" "4.3" "6.2" ...
 $ Tested..population..   : chr  "281" "63" "774" "201" ...
 $ Confirmed..population..: chr  "27.4" "31.7" "32.9" "12.5" ...
 $ Ref.                   : chr  "[431][432]" "[328][329]" "[430]" "[396][397]" ...
 $ Positivity_Rate        : num  9.76 5.03 4.25 6.21 10.71 ...
 $ Tests_Per_Case         : num  10.24 19.87 23.5 16.1 9.33 ...
 - attr(*, "na.action")= 'omit' Named int [1:17] 23 28 29 42 45 49 51 64 67 74 ...
  ..- attr(*, "names")= chr [1:17] "23" "28" "29" "42" ...


#Step 2: Calculate Total Tested Cases Worldwide

In [53]:
%%R
total_tested <- sum(covid_df$Tested, na.rm = TRUE)

total_tested

[1] 5305175205


#Step 3: Calculate Total Confirmed Cases Worldwide

In [54]:
%%R
total_confirmed <- sum(covid_df$Confirmed.cases., na.rm = TRUE)

total_confirmed

[1] 423654942


#Step 4: Calculate Worldwide Positive Ratio

In [55]:
%%R
positive_ratio <- total_confirmed / total_tested

positive_ratio

[1] 0.07985692


#Step 5: Convert to Percentage

In [56]:
%%R
positive_percentage <- positive_ratio * 100

positive_percentage

[1] 7.985692


#Step 6: Display Results

In [57]:
%%R
cat("Worldwide COVID Testing Statistics\n")
cat("-----------------------------------\n")
cat("Total Tested Cases:", format(total_tested, big.mark=","), "\n")
cat("Total Confirmed Cases:", format(total_confirmed, big.mark=","), "\n")
cat("Worldwide Positive Ratio:", round(positive_ratio, 4), "\n")
cat("Worldwide Positive Percentage:", round(positive_percentage, 2), "%\n")

Worldwide COVID Testing Statistics
-----------------------------------
Total Tested Cases: 5,305,175,205 
Total Confirmed Cases: 423,654,942 
Worldwide Positive Ratio: 0.0799 
Worldwide Positive Percentage: 7.99 %


#Alternative Solution Using dplyr

In [58]:
%%R
library(dplyr)

covid_summary <- covid_df %>%
  summarise(
    Total_Tested = sum(Tested, na.rm = TRUE),
    Total_Confirmed = sum(Confirmed.cases., na.rm = TRUE) # Corrected column name
  ) %>%
  mutate(
    Positive_Ratio = Total_Confirmed / Total_Tested,
    Positive_Percentage = Positive_Ratio * 100
  )

covid_summary

  Total_Tested Total_Confirmed Positive_Ratio Positive_Percentage
1   5305175205       423654942     0.07985692            7.985692


#Interpretation

- A positive ratio of 0.10 means approximately 1 out of every 10 tests returned positive.
- A positive percentage of 10% indicates the proportion of all COVID-19 tests that resulted in confirmed cases worldwide.
- Lower positivity rates generally indicate broader testing coverage, while higher rates may suggest under-testing or widespread transmission.

#TASK 6: Get a country list which reported their testing data


The goal of task 6 is to get a catalog or sorted list of countries who have reported their COVID-19 testing data

#Step 1: Extract the Country Column

Assuming the data frame contains a column named Country.or.region:

In [59]:
%%R
country_list <- covid_df$Country.or.region

#Step 2: Remove Missing Values

In [60]:
%%R
country_list <- na.omit(country_list)

#Step 3: Remove Duplicate Countries

In [61]:
%%R
country_list <- unique(country_list)

#Step 4: Sort the Country List Alphabetically

In [62]:
%%R
country_list <- sort(country_list)

#Step 5: Display the Country List

In [63]:
%%R
print(country_list)

  [1] ".mw-Parser-Output .reflist-Columns-2{column-Width:30em}.mw-Parser-Output .reflist-Columns-3{column-Width:25em}body.skin-Vector-2022 .mw-Parser-Output .reflist-Columns-2{column-Width:27em}body.skin-Vector-2022 .mw-Parser-Output .reflist-Columns-3{column-Width:22.5em}.mw-Parser-Output .references[data-Mw-Group=upper-Alpha]{list-Style-Type:upper-Alpha}.mw-Parser-Output .references[data-Mw-Group=upper-Roman]{list-Style-Type:upper-Roman}.mw-Parser-Output .references[data-Mw-Group=lower-Alpha]{list-Style-Type:lower-Alpha}.mw-Parser-Output .references[data-Mw-Group=lower-Greek]{list-Style-Type:lower-Greek}.mw-Parser-Output .references[data-Mw-Group=lower-Roman]{list-Style-Type:lower-Roman}.mw-Parser-Output Div.reflist-Liststyle-Upper-Alpha .references{list-Style-Type:upper-Alpha}.mw-Parser-Output Div.reflist-Liststyle-Upper-Roman .references{list-Style-Type:upper-Roman}.mw-Parser-Output Div.reflist-Liststyle-Lower-Alpha .references{list-Style-Type:lower-Alpha}.mw-Parser-Output Div.refl

#Step 6: Count the Number of Reporting Countries

In [64]:
%%R
num_countries <- length(country_list)

cat("Number of countries reporting testing data:", num_countries)

Number of countries reporting testing data: 156

#Create a Data Frame of Reporting Countries

In [65]:
%%R
countries_df <- data.frame(
  Country = country_list
)

head(countries_df)

#Export the Country List

In [66]:
%%R
write.csv(
  countries_df,
  "reporting_countries.csv",
  row.names = FALSE
)

#Expected Outcome

The resulting dataset:

- Contains one row per country/region that reported COVID-19 testing data.

- Has no duplicates or missing values.

- Is sorted alphabetically for easier lookup.

- Can be exported as reporting_countries.csv for further analysis or visualization.

#TASK 7: Identify countries names with a specific pattern

The goal of task 7 is using a regular expression to find any countires start with United

#Step 1: Review the Country List

Assume you already have:

In [67]:
%%R
country_list <- covid_df$Country.or.region

#Step 2: Use a Regular Expression

The regex pattern:

In [68]:
"^United"

'^United'

#Step 3: Find Matching Countries

In [69]:
%%R
united_countries <- grep(
  "^United",
  country_list,
  value = TRUE
)

united_countries

[1] "United States"        "United Kingdom"       "United Arab Emirates"


Alternative Using grepl()

In [70]:
%%R
united_countries <- country_list[
  grepl("^United", country_list)
]

united_countries

[1] "United States"        "United Kingdom"       "United Arab Emirates"


#Step 4: Sort the Results

In [71]:
%%R
united_countries <- sort(unique(united_countries))

print(united_countries)

[1] "United Arab Emirates" "United Kingdom"       "United States"       


#Step 5: Count the Matches

In [72]:
%%R
cat(
  "Number of countries starting with 'United':",
  length(united_countries)
)

Number of countries starting with 'United': 3

#Expected Output

Depending on the Wikipedia data, you may see:

[1] "United Arab Emirates"

[2] "United Kingdom"

[3] "United States"

Countries starting with 'United': 3

TASK 8: Pick two countries you are interested, and then review their testing data


The goal of task 8 is to compare the COVID-19 test data between two countires, you will need to select two rows from the dataframe, and select country, confirmed, confirmed-population-ratio columns

#Step 1: Choose Two Countries

For example:

United States

India

You can replace these with any two countries of interest.

#Step 2: Select the Required Columns

In [73]:
%%R
selected_data <- covid_df[
  covid_df$Country.or.region %in% c("United States", "India"),
  c("Country.or.region",
    "Confirmed.cases.",
    "Confirmed..population..")
]

selected_data

    Country.or.region Confirmed.cases. Confirmed..population..
166     United States         90749469                    27.4
73              India         43585554                    31.7


#Step 3: Using dplyr (Alternative)

In [74]:
%%R
library(dplyr)

selected_data <- covid_df %>%
  filter(Country.or.region %in%
           c("United States", "India")) %>%
  select(
    Country.or.region,
    Confirmed.cases.,
    Confirmed..population..
  )

selected_data

  Country.or.region Confirmed.cases. Confirmed..population..
1     United States         90749469                    27.4
2             India         43585554                    31.7


#Step 4: Display Results

In [75]:
%%R
print(selected_data)

  Country.or.region Confirmed.cases. Confirmed..population..
1     United States         90749469                    27.4
2             India         43585554                    31.7


#Step 5: Compare the Two Countries

In [76]:
%%R
usa_ratio <- selected_data$Confirmed..population..[
  selected_data$Country.or.region == "United States"
]

india_ratio <- selected_data$Confirmed..population..[
  selected_data$Country.or.region == "India"
]

cat("US Confirmed/Population Ratio:", usa_ratio, "\n")
cat("India Confirmed/Population Ratio:", india_ratio, "\n")

US Confirmed/Population Ratio: 27.4 
India Confirmed/Population Ratio: 31.7 


#Interpretation

After extracting the two rows:

Confirmed Cases shows the total number of reported COVID-19 infections.
Confirmed Population Ratio indicates the proportion of the population that has been confirmed infected.

For example:

A ratio of 0.31 means approximately 31% of the population had confirmed COVID-19 cases.
A ratio of 0.03 means approximately 3% of the population had confirmed COVID-19 cases.

This comparison helps evaluate the relative impact of COVID-19 across countries while accounting for differences in population size.

#TASK 9: Compare which one of the selected countries has a larger ratio of confirmed cases to population

The goal of task 9 is to find out which country you have selected before has larger ratio of confirmed cases to population, which may indicate that country has higher COVID-19 infection risk

#Step 1: Retrieve the Two Countries' Data

Using the same countries selected in Task 8 (e.g., United States and India):

In [77]:
%%R
comparison_df <- covid_df[
  covid_df$Country.or.region %in% c("United States", "India"),
  c("Country.or.region",
    "Confirmed.cases.",
    "Confirmed..population..")
]

comparison_df

    Country.or.region Confirmed.cases. Confirmed..population..
166     United States         90749469                    27.4
73              India         43585554                    31.7


#Step 2: Find the Country with the Highest Ratio

In [78]:
%%R
highest_ratio_country <- comparison_df[
  which.max(comparison_df$Confirmed..population..),
]

highest_ratio_country

   Country.or.region Confirmed.cases. Confirmed..population..
73             India         43585554                    31.7


#Step 3: Display the Result

In [79]:
%%R
cat(
  "Country with the higher confirmed-to-population ratio:",
  highest_ratio_country$Country.or.region,
  "\n"
)

cat(
  "Ratio:",
  round(as.numeric(highest_ratio_country$Confirmed..population..), 4)
)

Country with the higher confirmed-to-population ratio: India 
Ratio: 31.7

#Alternative Using dplyr

In [80]:
%%R
library(dplyr)

highest_ratio_country <- comparison_df %>%
  arrange(desc(Confirmed..population..)) %>%
  slice(1)

highest_ratio_country

  Country.or.region Confirmed.cases. Confirmed..population..
1             India         43585554                    31.7


#Interpretation

The United States has a larger proportion of confirmed COVID-19 cases relative to its population than India in the dataset. Therefore, based on the reported data, the United States shows a higher recorded infection prevalence. However, this metric should be interpreted carefully because differences in testing rates, reporting practices, demographics, and public health policies can influence the observed ratio.

#TASK 10: Find countries with confirmed to population ratio rate less than a threshold


The goal of task 10 is to find out which countries have the confirmed to population ratio less than 1%, it may indicate the risk of those countries are relatively low

#Step 1: Define the Threshold

In [81]:
%%R
threshold <- 0.01

# Step 2: Filter Countries Below the Threshold

Using base R:

In [82]:
%%R
low_risk_countries <- covid_df[
  covid_df$Confirmed..population.. < threshold,
  c("Country.or.region",
    "Confirmed..population..")
]

low_risk_countries

#Step 3: Sort by Ratio (Ascending)

In [85]:
%%R
low_risk_countries <- low_risk_countries[
  order(low_risk_countries$Confirmed..population..),
]

head(low_risk_countries)

#Step 4: Count the Number of Countries

In [88]:
%%R
num_countries <- nrow(low_risk_countries)

cat(
  "Number of countries with confirmed/population ratio below 1%:",
  num_countries
)

Number of countries with confirmed/population ratio below 1%: 4

#Step 5: Display the Results

In [90]:
%%R
print(low_risk_countries)

#Alternative Using dplyr

In [93]:
%%R
library(dplyr)

low_risk_countries <- covid_df %>%
  filter(Confirmed..population.. < 0.01) %>% # Corrected column name
  select(
    Country.or.region,
    Confirmed..population..
  ) %>%
  arrange(Confirmed..population..)

low_risk_countries

  Country.or.region Confirmed..population..
1       North Korea                 0.00000
2              Laos                 0.00063
3          China[c]                 0.00610


#Export the Results

In [95]:
%%R
write.csv(
  low_risk_countries,
  "countries_below_1_percent_ratio.csv",
  row.names = FALSE
)

#Interpretation

Countries with a Confirmed-to-Population Ratio below 1% have reported fewer confirmed COVID-19 cases relative to their population size.

A ratio below 1% may suggest:

Lower reported infection prevalence.
Effective containment measures.
Lower population exposure.
Differences in testing or reporting practices.

Therefore, while these countries may appear to have relatively lower COVID-19 impact based on confirmed cases, the ratio should be interpreted alongside testing coverage and reporting quality.